# 05a · Convert AT2 cells to a Seurat object

First half of the consensus NMF step (paper Methods: *Non-malignant lung single cell data processing*). Runs in R 3.6 with Seurat and SeuratDisk.

**Input** (File 04): `intermediate_files/lung/02_AT2_normal.h5ad`, `02_at2_var.csv`, `02_at2_obs.csv`

**Steps**
1. Convert the AnnData file to `h5seurat` and load it. The file holds the log-normalised expression matrix, which SeuratDisk copies into both the `counts` and `data` slots; File 05b uses the `data` slot.
2. Restore gene symbols and gene metadata from `02_at2_var.csv`.
3. Add cell metadata from `02_at2_obs.csv`.

**Output**: `AT2_seurat_object_figs.rds`, used by File 05b.

## Setup

In [1]:
library(Seurat)
library(SeuratDisk)

Attaching SeuratObject

Attaching sp

Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat



In [2]:
# Paths: edit to match your environment
data_dir <- "intermediate_files/lung"
out_rds <- "AT2_seurat_object_figs.rds"

## 1. Convert and load

In [3]:
Convert(file.path(data_dir, "02_AT2_normal.h5ad"), dest = "h5seurat", overwrite = TRUE)

Warning message:
“Unknown file type: h5ad”
Warning message:
“'assay' not set, setting to 'RNA'”
Creating h5Seurat file for version 3.1.5.9900

Adding X as data

Adding X as counts

Adding meta.features from var



In [4]:
AT2 <- LoadH5Seurat(file.path(data_dir, "02_AT2_normal.h5seurat"), meta.data = FALSE, misc = FALSE)

Validating h5Seurat file

Initializing RNA with data

Adding counts for RNA

Adding feature-level metadata for RNA

Adding command information

Adding tool-specific results



## 2. Gene symbols and gene metadata

In [5]:
at2_var <- read.csv(file.path(data_dir, "02_at2_var.csv"), row.names = 1)

# Check that the CSV genes are present in the Seurat object
if (!all(rownames(at2_var) %in% rownames(AT2@assays$RNA))) {
  stop("Not all gene names in the CSV file are present in the Seurat object.")
}

# Gene symbols from the 'genes' column
new_gene_names <- at2_var$genes

# The new names must be unique and match the number of features
if (length(unique(new_gene_names)) != length(new_gene_names)) {
  stop("The new gene names are not unique.")
}
if (length(new_gene_names) != nrow(AT2@assays$RNA@counts)) {
  stop("The number of new gene names does not match the number of features in the Seurat object.")
}

# Update the feature names in the Seurat object
rownames(AT2@assays$RNA@counts) <- new_gene_names
rownames(AT2@assays$RNA@data) <- new_gene_names
rownames(AT2@assays$RNA@meta.features) <- new_gene_names

# Add all columns of at2_var as gene-level metadata
AT2@assays$RNA@meta.features <- cbind(AT2@assays$RNA@meta.features, at2_var)

head(AT2@assays$RNA@meta.features)

,index,feature_name,feature_reference,feature_biotype,feature_length,feature_type,ensid,feature_base,mt,ribo,⋯,total_counts,log1p_total_counts,n_cells,highly_variable,means,dispersions,dispersions_norm,highly_variable_nbatches,highly_variable_intersection,genes
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>
A1BG,A1BG,A1BG_ENSG00000121410,NCBITaxon:9606,gene,2134,protein_coding,ENSG00000121410,A1BG,False,False,⋯,267808,12.498030,109535,False,0.0663783838,0.38974086,0.36673940,48,False,A1BG
A1BG-AS1,A1BG-AS1,A1BG-AS1,NCBITaxon:9606,gene,1667,lncRNA,ENSG00000268895,A1BG-AS1,False,False,⋯,37133,10.522288,20621,False,0.0110682097,0.24787680,0.07715406,32,False,A1BG-AS1
A1CF,A1CF,A1CF_ENSG00000148584,NCBITaxon:9606,gene,2211,protein_coding,ENSG00000148584,A1CF,False,False,⋯,1280,7.155397,489,False,0.0003904931,-0.04833500,-0.13314228,8,False,A1CF
A2M,A2M,A2M_ENSG00000175899,NCBITaxon:9606,gene,590,protein_coding,ENSG00000175899,A2M,False,False,⋯,2169287,14.589910,192273,True,0.3242143838,1.72704112,2.21391920,321,False,A2M
A2M-AS1,A2M-AS1,A2M-AS1,NCBITaxon:9606,gene,2551,lncRNA,ENSG00000245105,A2M-AS1,False,False,⋯,15750,9.664660,9727,False,0.0083671011,0.28704440,0.18237510,66,False,A2M-AS1
A2ML1,A2ML1,A2ML1_ENSG00000166535,NCBITaxon:9606,gene,592,protein_coding,ENSG00000166535,A2ML1,False,False,⋯,22099,10.003333,9581,False,0.0080369597,-0.04637315,-0.17286178,37,False,A2ML1


In [6]:
# Extract the new feature names from the 'genes' column
new_feature_names <- AT2@assays$RNA@meta.features$genes

# The new feature names must be unique and match the number of features
if (length(unique(new_feature_names)) != length(new_feature_names)) {
  stop("The new feature names are not unique.")
}
if (length(new_feature_names) != nrow(AT2@assays$RNA@counts)) {
  stop("The number of new feature names does not match the number of features in the Seurat object.")
}

# Update the row names of the Seurat object
rownames(AT2@assays$RNA@counts) <- new_feature_names
rownames(AT2@assays$RNA@data) <- new_feature_names
rownames(AT2@assays$RNA@meta.features) <- new_feature_names

## 3. Cell metadata

In [7]:
at2_obs <- read.csv(file.path(data_dir, "02_at2_obs.csv"), row.names = 1)

# Check that the CSV cells are present in the Seurat object
if (!all(rownames(at2_obs) %in% colnames(AT2))) {
  stop("Not all cell names in the CSV file are present in the Seurat object.")
}

AT2 <- AddMetaData(object = AT2, metadata = at2_obs)

head(AT2@meta.data)

,suspension_type,donor_id,is_primary_data,assay_ontology_term_id,cell_type_ontology_term_id,development_stage_ontology_term_id,disease_ontology_term_id,self_reported_ethnicity_ontology_term_id,tissue_ontology_term_id,organism_ontology_term_id,⋯,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,total_counts_ribo,log1p_total_counts_ribo,pct_counts_ribo,total_counts_hb,log1p_total_counts_hb,pct_counts_hb
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
GCATACAGTTGACGTT_F01641,cell,F01641,True,EFO:0011025,CL:0002063,HsapDv:0000148,PATO:0000461,HANCESTRO:0005,UBERON:0008946,NCBITaxon:9606,⋯,77.34831,603,6.403574,9.0337080,992,6.900731,14.8614230,1,0.6931472,0.014981274
D062_ACTATGGTCTCGCGTT-sub_wang_sub_batch3,nucleus,GSM4906348,False,EFO:0009922,CL:0002063,HsapDv:0000262,PATO:0000461,unknown,UBERON:0002048,NCBITaxon:9606,⋯,68.03101,28,3.367296,0.8682171,21,3.091042,0.6511627,2,1.0986123,0.062015500
CTCGTCACAGCGATCC_NML3_tsukui,cell,NML3_Lin,True,EFO:0009899,CL:0002063,unknown,PATO:0000461,unknown,UBERON:0002048,NCBITaxon:9606,⋯,79.00707,54,4.007333,0.7955215,1410,7.252054,20.7719500,0,0.0000000,0.000000000
CGGTTAACACATGGGA_SC20,cell,SC20,True,EFO:0009899,CL:0002063,HsapDv:0000151,PATO:0000461,HANCESTRO:0010,UBERON:0008946,NCBITaxon:9606,⋯,68.85155,866,6.765039,4.0087028,3385,8.127404,15.6691190,1,0.6931472,0.004628987
LAP88_AAGGAATTCTAGTTCT-1_duong,nucleus,Duong_lungMAP_unpubl_sample_3,True,EFO:0009922,CL:0002063,HsapDv:0000131,PATO:0000461,HANCESTRO:0010,UBERON:0001005,NCBITaxon:9606,⋯,63.88759,0,0.000000,0.0000000,33,3.526361,0.7728337,0,0.0000000,0.000000000
P1_4_ATTGGACAGACGCACA,cell,distal 1a,False,EFO:0009899,CL:0002063,HsapDv:0000169,PATO:0000461,HANCESTRO:0005,UBERON:0008946,NCBITaxon:9606,⋯,78.70610,364,5.899898,4.7004130,788,6.670766,10.1756200,0,0.0000000,0.000000000


## 4. Save

In [8]:
saveRDS(AT2, file = out_rds)